# Управление портфелем: эффективные портфели на российских акциях

Портфельный анализ 30 российских акций (голубые фишки MOEX) за период 2015-2025.
Применяем теорию Марковица: строим границы эффективных портфелей при разных режимах ограничений,
оцениваем бета-коэффициенты рыночной модели, сравниваем подходы к оценке ковариационной матрицы.

**Данные:** ежедневные цены закрытия 30 акций с MOEX ISS API, бенчмарк IMOEX, безрисковая ставка -- ключевая ставка ЦБ РФ.

**Методы:** historical covariance (rolling 252d), market model (beta-based), EWMA, expanding window.

## Настройка окружения

Для работы в Google Colab загрузите папку `data/` рядом с этим notebook (или подключите Google Drive через `drive.mount`).

In [ ]:
# установка пакетов (для Colab; локально можно пропустить)
# !pip install -q pandas numpy matplotlib seaborn scipy statsmodels pyarrow

import numpy as np
import pandas as pd
import pickle
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
import seaborn as sns

from scipy.optimize import minimize
import statsmodels.api as sm

warnings.filterwarnings('ignore', category=FutureWarning)

print('Импорт завершён')

In [ ]:
# воспроизводимость
np.random.seed(42)

# matplotlib: русские подписи через DejaVu Sans
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150

_available_fonts = [f.name for f in matplotlib.font_manager.fontManager.ttflist]
if 'DejaVu Sans' in _available_fonts:
    plt.rcParams['font.family'] = ['DejaVu Sans']

# пути к данным -- предполагаем, что data/ лежит рядом с notebook
DATA_DIR = Path('data')
PROCESSED = DATA_DIR / 'processed'
RAW_DIR = DATA_DIR / 'raw'
META_DIR = DATA_DIR / 'meta'

# проверка, что данные на месте
assert PROCESSED.exists(), f'Папка {PROCESSED} не найдена -- загрузите data/ рядом с notebook'
print(f'Данные: {PROCESSED}')

---
## 1. Задача 1 -- Сбор данных

Собрали ежедневные цены закрытия 30 российских акций с MOEX ISS API за период 01.01.2015 -- 31.12.2025. Учтены 3 корпоративных действия (сплиты), цены скорректированы backward adjustment. Бенчмарк -- индекс IMOEX, безрисковая ставка -- ключевая ставка ЦБ РФ.

In [ ]:
# справочник тикеров
instruments = pd.read_csv(META_DIR / 'instruments.csv')
instruments

In [ ]:
# скорректированные цены закрытия
prices = pd.read_parquet(PROCESSED / 'prices_adjusted.parquet')
print(f'Размер: {prices.shape[0]} дней x {prices.shape[1]} тикеров')
print(f'Период: {prices.index.min().date()} -- {prices.index.max().date()}')
prices.head()

In [ ]:
# нормализованные цены (база = 100 на первый день)
prices_norm = prices / prices.iloc[0] * 100

fig, ax = plt.subplots(figsize=(14, 7))
for col in prices_norm.columns:
    ax.plot(prices_norm.index, prices_norm[col], linewidth=0.8, alpha=0.7, label=col)

ax.set_ylabel('Нормализованная цена (база = 100)')
ax.set_title('Динамика цен 30 акций, 2015-2025')
ax.legend(fontsize=6, ncol=5, loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Данные покрывают весь требуемый период. Видно приостановку торгов в феврале-марте 2022, а также сильную дифференциацию динамики: золотодобытчик PLZL и нефтегазовые компании ведут себя по-разному.

---
## 2. Задачи 2-3 -- Доходности и ковариационные матрицы

Рассчитаны простые дневные доходности (для Марковица) и 7 методов оценки ковариационных матриц: rolling 252d/63d/21d, expanding window, EWMA с тремя значениями lambda. Для основного анализа выбрана rolling 252d -- стандартный годовой горизонт.

In [ ]:
# дневные доходности
returns = pd.read_parquet(PROCESSED / 'returns_daily.parquet')
print(f'Доходности: {returns.shape[0]} дней, {returns.shape[1]} тикеров')

# описательная статистика (аннуализированная)
stats = pd.DataFrame({
    'mean (ann.)': returns.mean() * 252,
    'std (ann.)': returns.std() * np.sqrt(252),
    'min': returns.min(),
    'max': returns.max(),
    'skew': returns.skew(),
    'kurt': returns.kurtosis(),
})
stats = stats.round(4)
stats.sort_values('mean (ann.)', ascending=False)

In [ ]:
# ковариационная матрица (rolling 252d, конец периода)
selected_cov = pd.read_parquet(PROCESSED / 'selected_cov.parquet')

# корреляционная матрица
std_diag = np.sqrt(np.diag(selected_cov.values))
corr_matrix = selected_cov.values / np.outer(std_diag, std_diag)
corr_df = pd.DataFrame(corr_matrix, index=selected_cov.index, columns=selected_cov.columns)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_df, vmin=-0.2, vmax=1.0, cmap='RdBu_r', square=True,
            linewidths=0.3, ax=ax,
            cbar_kws={'shrink': 0.8, 'label': 'Корреляция'})
ax.set_title('Корреляционная матрица (rolling 252d, конец 2025)', fontsize=13)
ax.tick_params(axis='both', labelsize=7)
plt.tight_layout()
plt.show()

Выбрано скользящее окно 252 дня (конец 30.12.2025) как основной метод оценки ожидаемых доходностей и ковариаций. Это стандартный годовой горизонт, обеспечивающий баланс между актуальностью и статистической устойчивостью.

---
## 3. Задачи 4-8 -- Граница эффективных портфелей

Построены границы EF для 4 режимов ограничений:
1. **Unrestricted** -- без ограничений (короткие продажи разрешены)
2. **Short 25%** -- минимальный вес -25%, максимальный 100%
3. **Long-only** -- без коротких продаж (0 <= w <= 1)
4. **Min 2%** -- каждая акция минимум 2% (2% <= w <= 100%)

Безрисковая ставка rf = 16% (ключевая ставка ЦБ на конец 2025).

Ниже -- функции оптимизации, которые используются далее.

In [ ]:
# --- Модуль оптимизации (из step3_optimizer.py, встроен в notebook) ---

def portfolio_variance(w, cov):
    """Дисперсия портфеля: w^T @ Sigma @ w."""
    return w @ cov @ w


def portfolio_volatility(w, cov):
    """Волатильность портфеля (годовая)."""
    return np.sqrt(w @ cov @ w)


def portfolio_return(w, mu):
    """Ожидаемая доходность портфеля."""
    return w @ mu


def portfolio_sharpe(w, mu, cov, rf):
    """Коэффициент Шарпа портфеля."""
    ret = w @ mu
    vol = np.sqrt(w @ cov @ w)
    if vol < 1e-10:
        return 0.0
    return (ret - rf) / vol


def optimize_for_target(mu, cov, target_return, bounds=None):
    """
    Найти портфель с минимальной дисперсией для заданной целевой доходности.
    min w^T Sigma w  s.t.  w^T mu = target, sum(w) = 1
    """
    n = len(mu)
    x0 = np.ones(n) / n

    constraints = [
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0},
        {'type': 'eq', 'fun': lambda w: w @ mu - target_return},
    ]

    result = minimize(
        portfolio_variance, x0, args=(cov,),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'ftol': 1e-12, 'maxiter': 1000}
    )
    return result


def find_gmvp(mu, cov, bounds=None):
    """
    Global Minimum Variance Portfolio -- портфель с минимальным риском.
    """
    n = len(mu)
    x0 = np.ones(n) / n

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]

    result = minimize(
        portfolio_variance, x0, args=(cov,),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'ftol': 1e-12, 'maxiter': 1000}
    )

    if not result.success:
        print(f'GMVP: optimizer failed -- {result.message}')

    w = result.x
    return {
        'weights': w,
        'return': w @ mu,
        'std': np.sqrt(w @ cov @ w),
        'success': result.success,
    }


def find_tangency(mu, cov, rf, bounds=None):
    """
    Tangency portfolio -- максимальный Sharpe ratio.
    Несколько стартовых точек + аналитическая проверка для unrestricted.
    """
    n = len(mu)
    excess = mu - rf
    best = None

    top_idx = np.argsort(excess)[::-1]
    starts = [np.ones(n) / n]
    for k in [1, 3, 5, 10]:
        x0 = np.zeros(n)
        if bounds is not None:
            lb = np.array([b[0] if b[0] is not None else -10.0 for b in bounds])
            x0[:] = np.maximum(lb, 0.0)
            remaining = 1.0 - x0.sum()
            if remaining > 0:
                x0[top_idx[:k]] += remaining / k
            else:
                x0 = np.ones(n) / n
        else:
            x0[top_idx[:k]] = 1.0 / k
        starts.append(x0)

    def neg_sharpe(w):
        ret = w @ mu
        vol = np.sqrt(w @ cov @ w)
        if vol < 1e-10:
            return 0.0
        return -(ret - rf) / vol

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]

    for x0 in starts:
        result = minimize(
            neg_sharpe, x0,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints,
            options={'ftol': 1e-12, 'maxiter': 2000}
        )
        if result.success:
            w = result.x
            ret = w @ mu
            vol = np.sqrt(w @ cov @ w)
            sharpe = (ret - rf) / vol if vol > 1e-10 else 0.0
            if best is None or sharpe > best['sharpe']:
                best = {
                    'weights': w.copy(),
                    'return': ret,
                    'std': vol,
                    'sharpe': sharpe,
                    'success': True,
                }

    # для unrestricted: аналитическое решение
    if bounds is None:
        Sigma_inv = np.linalg.inv(cov)
        w_raw = Sigma_inv @ excess
        denom = np.sum(w_raw)
        if abs(denom) > 1e-10:
            w_an = w_raw / denom
            ret_an = w_an @ mu
            vol_an = np.sqrt(w_an @ cov @ w_an)
            sharpe_an = (ret_an - rf) / vol_an if vol_an > 1e-10 else 0.0
            if best is None or sharpe_an > best['sharpe']:
                best = {
                    'weights': w_an.copy(),
                    'return': ret_an,
                    'std': vol_an,
                    'sharpe': sharpe_an,
                    'success': True,
                }

    if best is None:
        gmvp = find_gmvp(mu, cov, bounds)
        gmvp['sharpe'] = (gmvp['return'] - rf) / gmvp['std']
        return gmvp

    return best


def find_max_return(mu, cov, bounds=None):
    """Портфель с максимальной доходностью при заданных ограничениях."""
    n = len(mu)
    x0 = np.ones(n) / n

    def neg_return(w):
        return -(w @ mu)

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]

    result = minimize(
        neg_return, x0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'ftol': 1e-12, 'maxiter': 1000}
    )

    w = result.x
    return {
        'weights': w,
        'return': w @ mu,
        'std': np.sqrt(w @ cov @ w),
        'success': result.success,
    }


def build_efficient_frontier(mu, cov, rf, n_points=200, bounds=None,
                             mu_min=None, mu_max=None):
    """
    Построить границу эффективных портфелей.
    Tangency определяется как точка с max Sharpe на построенной границе.
    """
    n = len(mu)

    gmvp = find_gmvp(mu, cov, bounds)
    if mu_min is None:
        mu_min = gmvp['return']

    if mu_max is None:
        max_ret = find_max_return(mu, cov, bounds)
        mu_max = max_ret['return']

    targets = np.linspace(mu_min, mu_max, n_points)

    records = []
    weights_list = []

    for target in targets:
        res = optimize_for_target(mu, cov, target, bounds)
        if res.success:
            w = res.x
            ret = w @ mu
            vol = np.sqrt(w @ cov @ w)
            sharpe = (ret - rf) / vol if vol > 1e-10 else 0.0
            records.append({
                'target_return': target,
                'portfolio_return': ret,
                'portfolio_std': vol,
                'sharpe': sharpe,
            })
            weights_list.append(w.copy())

    frontier_df = pd.DataFrame(records)
    frontier_weights = np.array(weights_list) if weights_list else np.empty((0, n))

    if len(frontier_df) > 0:
        best_idx = frontier_df['sharpe'].idxmax()
        tangency = {
            'weights': frontier_weights[best_idx].copy(),
            'return': frontier_df.iloc[best_idx]['portfolio_return'],
            'std': frontier_df.iloc[best_idx]['portfolio_std'],
            'sharpe': frontier_df.iloc[best_idx]['sharpe'],
            'success': True,
        }
    else:
        tangency = gmvp.copy()
        tangency['sharpe'] = (tangency['return'] - rf) / tangency['std']

    key_portfolios = {'gmvp': gmvp, 'tangency': tangency}

    return frontier_df, frontier_weights, key_portfolios


print('Функции оптимизации загружены')

In [ ]:
# загрузка выбранных параметров (rolling 252d, конец периода)
selected_mu_df = pd.read_parquet(PROCESSED / 'selected_mu.parquet')
selected_cov_df = pd.read_parquet(PROCESSED / 'selected_cov.parquet')
selected_rf_df = pd.read_parquet(PROCESSED / 'selected_rf.parquet')

tickers = selected_mu_df['ticker'].tolist()
mu = selected_mu_df['expected_return'].values
Sigma = selected_cov_df.values
rf = selected_rf_df['rf_annual'].iloc[0]

n = len(tickers)
print(f'Тикеров: {n}, rf = {rf:.2%}')
print(f'Доходности: от {mu.min():.2%} до {mu.max():.2%}')

In [ ]:
# загрузка предрассчитанных границ
ef_unrestricted = pd.read_parquet(PROCESSED / 'ef_unrestricted.parquet')
ef_long_only = pd.read_parquet(PROCESSED / 'ef_long_only.parquet')
ef_short_25 = pd.read_parquet(PROCESSED / 'ef_short_25.parquet')
ef_min_2pct = pd.read_parquet(PROCESSED / 'ef_min_2pct.parquet')

# загрузка ключевых портфелей
with open(PROCESSED / 'ef_portfolios.pkl', 'rb') as f:
    ef_portfolios = pickle.load(f)

print(f'Frontiers loaded: unrestricted={len(ef_unrestricted)}, '
      f'long_only={len(ef_long_only)}, short_25={len(ef_short_25)}, '
      f'min_2pct={len(ef_min_2pct)}')

In [ ]:
# сводный график: 4 границы, CML, отдельные акции

REGIMES = ['unrestricted', 'short_25', 'long_only', 'min_2pct']
REGIME_LABELS = {
    'unrestricted': 'Без ограничений',
    'short_25':     'Шорт до 25%',
    'long_only':    'Только лонг',
    'min_2pct':     'Мин. вес 2%',
}
REGIME_COLORS = {
    'unrestricted': 'tab:blue',
    'short_25':     'tab:orange',
    'long_only':    'tab:green',
    'min_2pct':     'tab:red',
}

frontiers = {
    'unrestricted': ef_unrestricted,
    'short_25': ef_short_25,
    'long_only': ef_long_only,
    'min_2pct': ef_min_2pct,
}

vol_stocks = np.sqrt(np.diag(Sigma))

fig, ax = plt.subplots(figsize=(14, 10))

# определяем разумный диапазон осей
unr_std_max = ef_unrestricted['portfolio_std'].max()
x_max = max(unr_std_max, ef_long_only['portfolio_std'].max()) * 1.3
x_max = max(x_max, 0.50)

for regime in REGIMES:
    df = frontiers[regime]
    x = df['portfolio_std'].values * 100
    y = df['portfolio_return'].values * 100
    color = REGIME_COLORS[regime]
    label = REGIME_LABELS[regime]

    # для short_25 обрезаем хвост
    if regime == 'short_25':
        mask = df['portfolio_std'].values <= x_max
        x = x[mask]
        y = y[mask]

    ax.plot(x, y, color=color, linewidth=2, label=label, zorder=3)

# CML через tangency unrestricted
tang_unr = ef_portfolios['unrestricted']['tangency']
if tang_unr['std'] > 1e-10:
    slope = (tang_unr['return'] - rf) / tang_unr['std']
    x_cml = np.linspace(0, x_max, 200)
    y_cml = (rf + slope * x_cml) * 100
    ax.plot(x_cml * 100, y_cml, 'k--', linewidth=1.5, alpha=0.7, label='CML', zorder=2)

# отдельные акции
ax.scatter(vol_stocks * 100, mu * 100, c='gray', s=30, alpha=0.6, zorder=4)
for i, t in enumerate(tickers):
    ax.annotate(t, (vol_stocks[i] * 100, mu[i] * 100),
                fontsize=6, alpha=0.7, ha='left', va='bottom')

# GMVP и tangency каждого режима
for regime in REGIMES:
    color = REGIME_COLORS[regime]
    gmvp_r = ef_portfolios[regime]['gmvp']
    tang_r = ef_portfolios[regime]['tangency']
    ax.plot(gmvp_r['std'] * 100, gmvp_r['return'] * 100,
            'o', color=color, markersize=9, markeredgecolor='black',
            markeredgewidth=0.8, zorder=5)
    ax.plot(tang_r['std'] * 100, tang_r['return'] * 100,
            '*', color=color, markersize=14, markeredgecolor='black',
            markeredgewidth=0.8, zorder=5)

ax.plot(0, rf * 100, 'kD', markersize=8, zorder=5, label=f'rf = {rf*100:.1f}%')

ax.set_xlim(0, x_max * 100)
y_min_val = min(ef_unrestricted['portfolio_return'].min(),
                ef_min_2pct['portfolio_return'].min()) * 100
y_max_val = max(ef_unrestricted['portfolio_return'].max(),
                ef_long_only['portfolio_return'].max()) * 100
y_pad = (y_max_val - y_min_val) * 0.1
ax.set_ylim(y_min_val - y_pad, y_max_val + y_pad)

ax.set_xlabel('Годовая волатильность, %', fontsize=13)
ax.set_ylabel('Годовая доходность, %', fontsize=13)
ax.set_title('Сравнение границ эффективных портфелей', fontsize=15)

handles, labels_l = ax.get_legend_handles_labels()
handles.append(Line2D([], [], marker='o', color='gray', linestyle='None',
                      markersize=8, markeredgecolor='black', label='GMVP'))
handles.append(Line2D([], [], marker='*', color='gray', linestyle='None',
                      markersize=12, markeredgecolor='black', label='Tangency'))
ax.legend(handles=handles, loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# таблица ключевых портфелей
rows = []
for port_type, port_key in [('GMVP', 'gmvp'), ('Tangency', 'tangency')]:
    for regime in REGIMES:
        p = ef_portfolios[regime][port_key]
        w = p['weights']
        sharpe = p.get('sharpe', (p['return'] - rf) / p['std'] if p['std'] > 1e-10 else 0.0)
        rows.append({
            'Portfolio': port_type,
            'Regime': REGIME_LABELS[regime],
            'Return, %': f"{p['return'] * 100:.2f}",
            'Std, %': f"{p['std'] * 100:.2f}",
            'Sharpe': f'{sharpe:.3f}',
            'Max w, %': f'{w.max() * 100:.1f}',
            'Min w, %': f'{w.min() * 100:.1f}',
            'N(w>5%)': int(np.sum(np.abs(w) > 0.05)),
        })

table_df = pd.DataFrame(rows)
table_df

Sharpe ratio отрицательный для всех режимов, потому что rf = 16% превышает доходность GMVP. Это типичная ситуация для российского рынка в 2025 году при высокой ключевой ставке. По факту это означает, что депозит или ОФЗ доходнее любого портфеля на рынке акций при текущей конъюнктуре.

Вложенность границ подтверждается: unrestricted дает минимальный risk, далее short_25 и long_only, а min_2pct -- максимальный (самый ограниченный режим).

---
## 4. Задачи 9-10 -- Динамика EF

Строим серии efficient frontiers на разных контрольных датах (year-end, 2016-2025) для трёх методов оценки: rolling 252d, expanding window и EWMA. Это показывает, насколько граница нестабильна во времени.

In [ ]:
# функция для визуализации серии фронтиров
def plot_frontier_dynamics(result, title):
    """Overlay нескольких фронтиров на одном графике."""
    dates = result['dates']
    frontiers_list = result['frontiers']
    key_ports = result['key_portfolios']
    n_dates = len(dates)

    fig, ax = plt.subplots(figsize=(12, 8))

    norm = Normalize(vmin=0, vmax=max(n_dates - 1, 1))
    cmap = cm.coolwarm

    for i in range(n_dates):
        df = frontiers_list[i]
        if df.empty:
            continue

        color = cmap(norm(i))
        year_label = str(dates[i].year) if hasattr(dates[i], 'year') else str(dates[i])

        ax.plot(df['portfolio_std'] * 100, df['portfolio_return'] * 100,
                color=color, linewidth=1.5, alpha=0.8, label=year_label)

        # GMVP
        gmvp_i = key_ports[i]['gmvp']
        ax.scatter(gmvp_i['std'] * 100, gmvp_i['return'] * 100,
                   color=color, s=50, zorder=5, edgecolors='black', linewidths=0.5)

    ax.set_xlabel('Годовая волатильность, %', fontsize=12)
    ax.set_ylabel('Годовая доходность, %', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# динамика: rolling 252d
with open(PROCESSED / 'ef_dynamics_rolling_252d.pkl', 'rb') as f:
    ef_dyn_rolling = pickle.load(f)

plot_frontier_dynamics(ef_dyn_rolling, 'Динамика EF (rolling 252d, unrestricted)')

In [ ]:
# динамика: expanding window
with open(PROCESSED / 'ef_dynamics_expanding.pkl', 'rb') as f:
    ef_dyn_expanding = pickle.load(f)

plot_frontier_dynamics(ef_dyn_expanding, 'Динамика EF (expanding window, unrestricted)')

In [ ]:
# траектория GMVP по методам
methods_dyn = {
    'rolling_252d': ef_dyn_rolling,
    'expanding': ef_dyn_expanding,
}

# пробуем загрузить EWMA
for ewma_name in ['ewma_094', 'ewma_097', 'ewma_099']:
    ewma_path = PROCESSED / f'ef_dynamics_{ewma_name}.pkl'
    if ewma_path.exists():
        with open(ewma_path, 'rb') as f:
            methods_dyn[ewma_name] = pickle.load(f)

# таблица GMVP по годам
print('Траектория GMVP (rolling 252d)')
print(f'{"Дата":>12} | {"rf,%":>6} | {"Ret,%":>8} | {"Std,%":>8} | {"Sharpe":>8}')
print('-' * 55)

for i, d in enumerate(ef_dyn_rolling['dates']):
    kp = ef_dyn_rolling['key_portfolios'][i]
    gmvp_i = kp['gmvp']
    tang_i = kp['tangency']
    rf_i = ef_dyn_rolling['rf_values'][i]
    print(f'{str(d):>12} | {rf_i*100:>5.1f}% | {gmvp_i["return"]*100:>7.2f}% | '
          f'{gmvp_i["std"]*100:>7.2f}% | {tang_i["sharpe"]:>8.4f}')

Граница нестабильна во времени: в 2022 году (после приостановки торгов) она сильно сдвигается вправо, волатильность GMVP растёт. Expanding window сглаживает колебания по сравнению с rolling, но реагирует медленнее на структурные изменения рынка.

---
## 5. Задачи 11-12 -- Бета-коэффициенты

Рыночная модель: $r_i = \alpha_i + \beta_i \cdot r_m + \varepsilon_i$

Бенчмарк -- IMOEX, окно для OLS-регрессии -- 252 торговых дня (совпадает с окном для EF). Корректировка Блюма: $\beta_{adj} = \frac{2}{3} \beta_{hist} + \frac{1}{3}$.

In [ ]:
# загрузка бет
beta_hist_df = pd.read_parquet(PROCESSED / 'beta_historical.parquet')
beta_adj_df = pd.read_parquet(PROCESSED / 'beta_adjusted.parquet')

# объединяем в одну таблицу
beta_merged = beta_hist_df[['ticker', 'beta', 'r_squared', 't_stat_beta',
                            'se_beta', 'sigma_epsilon']].copy()
beta_merged = beta_merged.merge(beta_adj_df[['ticker', 'beta_adjusted']], on='ticker')
beta_merged = beta_merged.sort_values('beta', ascending=False)
beta_merged

In [ ]:
# bar chart: историческая vs скорректированная бета
beta_sorted = beta_merged.sort_values('beta')

fig, ax = plt.subplots(figsize=(10, 12))

y_pos = np.arange(len(beta_sorted))
width = 0.35

ax.barh(y_pos - width/2, beta_sorted['beta'], width,
        color='steelblue', label='$\\beta$ историческая')
ax.barh(y_pos + width/2, beta_sorted['beta_adjusted'], width,
        color='coral', label='$\\beta$ скорректированная (Blume)')

ax.set_yticks(y_pos)
ax.set_yticklabels(beta_sorted['ticker'], fontsize=9)
ax.axvline(x=1.0, color='black', linewidth=1.2, linestyle='--', label='$\\beta$ = 1')
ax.set_xlabel('$\\beta$', fontsize=12)
ax.set_title('Бета-коэффициенты 30 акций', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# сводка по регрессиям
n_signif = (beta_hist_df['t_stat_beta'].abs() > 2).sum()
betas = beta_hist_df['beta']
print(f'Значимых бет (|t| > 2): {n_signif} из {len(beta_hist_df)}')
print(f'beta: mean={betas.mean():.4f}, median={betas.median():.4f}, '
      f'min={betas.min():.4f}, max={betas.max():.4f}')
print(f'R2:   mean={beta_hist_df["r_squared"].mean():.4f}, '
      f'min={beta_hist_df["r_squared"].min():.4f}, '
      f'max={beta_hist_df["r_squared"].max():.4f}')

Все 30 бет значимы (t > 2). Диапазон бет 0.46-1.42, среднее примерно 1.0 -- что логично, т.к. все акции входят в IMOEX. Корректировка Блюма стягивает крайние значения к 1: высокие бета снижаются, низкие -- повышаются.

---
## 6. Задачи 13-15 -- EF на основе market model

Ковариационная матрица market model:

$$\Sigma_\beta = \beta \beta^\top \sigma^2_m + \text{diag}(\sigma^2_{\varepsilon})$$

Вместо оценки $n(n+1)/2 = 465$ параметров (историческая Sigma) нужно только $2n + 1 = 61$ (30 бет, 30 остаточных дисперсий, 1 рыночная дисперсия). Это делает оценку устойчивее, но вся корреляция проходит через единственный рыночный фактор.

In [ ]:
# реконструкция Sigma_beta из компонентов
bench_ret = pd.read_parquet(PROCESSED / 'benchmark_returns.parquet')
residuals = pd.read_parquet(PROCESSED / 'beta_residuals.parquet')

beta_values = beta_hist_df.set_index('ticker').loc[tickers, 'beta'].values
sigma_eps = beta_hist_df.set_index('ticker').loc[tickers, 'sigma_epsilon'].values

# рыночная дисперсия за окно бета (те же 252 дня)
r_m = bench_ret.loc[residuals.index, 'return_imoex'].values
sigma2_m = np.var(r_m, ddof=1)  # дневная

# аннуализация
sigma2_m_annual = sigma2_m * 252
sigma2_eps_annual = sigma_eps ** 2 * 252

# ковариационная матрица market model
systematic = np.outer(beta_values, beta_values) * sigma2_m_annual
idiosyncratic = np.diag(sigma2_eps_annual)
Sigma_beta = systematic + idiosyncratic

print(f'Sigma_beta: {Sigma_beta.shape}')
print(f'Symmetric: {np.allclose(Sigma_beta, Sigma_beta.T)}')
eigvals = np.linalg.eigvalsh(Sigma_beta)
print(f'PSD: {eigvals.min() >= -1e-12}')
print(f'Condition number: {eigvals.max() / eigvals.min():.1f}')

# доля систематического риска
sys_share = np.trace(systematic) / np.trace(Sigma_beta)
print(f'Доля систематического риска: {sys_share:.1%}')

In [ ]:
# сравнение корреляционных матриц: историческая vs beta-based
# историческая
cov_hist = selected_cov_df.loc[tickers, tickers].values
D_inv_hist = np.diag(1.0 / np.sqrt(np.diag(cov_hist)))
corr_hist = D_inv_hist @ cov_hist @ D_inv_hist

# beta-based
D_inv_beta = np.diag(1.0 / np.sqrt(np.diag(Sigma_beta)))
corr_beta = D_inv_beta @ Sigma_beta @ D_inv_beta

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

sns.heatmap(corr_hist, ax=axes[0], vmin=-0.2, vmax=1.0,
            xticklabels=tickers, yticklabels=tickers,
            cmap='RdBu_r', square=True, linewidths=0.3,
            cbar_kws={'shrink': 0.8, 'label': 'Корреляция'})
axes[0].set_title('Историческая корреляционная матрица\n(rolling 252d)', fontsize=12)
axes[0].tick_params(axis='both', labelsize=7)

sns.heatmap(corr_beta, ax=axes[1], vmin=-0.2, vmax=1.0,
            xticklabels=tickers, yticklabels=tickers,
            cmap='RdBu_r', square=True, linewidths=0.3,
            cbar_kws={'shrink': 0.8, 'label': 'Корреляция'})
axes[1].set_title('Корреляционная матрица market model\n(beta-based)', fontsize=12)
axes[1].tick_params(axis='both', labelsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# наложение EF: историческая vs beta-based
ef_beta_unr = pd.read_parquet(PROCESSED / 'ef_beta_unrestricted.parquet')
ef_beta_lo = pd.read_parquet(PROCESSED / 'ef_beta_long_only.parquet')

with open(PROCESSED / 'ef_beta_weights.pkl', 'rb') as f:
    w_beta = pickle.load(f)

with open(PROCESSED / 'ef_unrestricted_weights.pkl', 'rb') as f:
    w_hist_unr = pickle.load(f)
with open(PROCESSED / 'ef_long_only_weights.pkl', 'rb') as f:
    w_hist_lo = pickle.load(f)

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

# unrestricted
ax = axes[0]
ax.plot(ef_unrestricted['portfolio_std'] * 100, ef_unrestricted['portfolio_return'] * 100,
        'b-', linewidth=2, alpha=0.8, label='Historical')
ax.plot(ef_beta_unr['portfolio_std'] * 100, ef_beta_unr['portfolio_return'] * 100,
        'r-', linewidth=2, alpha=0.8, label='Market Model')

gmvp_h = w_hist_unr['gmvp']
gmvp_b = w_beta['unrestricted']['gmvp']
ax.scatter(gmvp_h['std']*100, gmvp_h['return']*100, marker='*', s=200, c='blue', zorder=5,
           label=f"GMVP hist ({gmvp_h['std']:.2%})")
ax.scatter(gmvp_b['std']*100, gmvp_b['return']*100, marker='*', s=200, c='red', zorder=5,
           label=f"GMVP beta ({gmvp_b['std']:.2%})")

ax.axhline(y=rf*100, color='orange', linestyle='--', alpha=0.5, label=f'rf = {rf:.0%}')
ax.set_xlabel('Волатильность, %', fontsize=11)
ax.set_ylabel('Доходность, %', fontsize=11)
ax.set_title('Unrestricted: Historical vs Market Model', fontsize=13)
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)

# long-only
ax = axes[1]
ax.plot(ef_long_only['portfolio_std'] * 100, ef_long_only['portfolio_return'] * 100,
        'b-', linewidth=2, alpha=0.8, label='Historical')
ax.plot(ef_beta_lo['portfolio_std'] * 100, ef_beta_lo['portfolio_return'] * 100,
        'r-', linewidth=2, alpha=0.8, label='Market Model')

gmvp_h_lo = w_hist_lo['gmvp']
gmvp_b_lo = w_beta['long_only']['gmvp']
ax.scatter(gmvp_h_lo['std']*100, gmvp_h_lo['return']*100, marker='*', s=200, c='blue', zorder=5,
           label=f"GMVP hist ({gmvp_h_lo['std']:.2%})")
ax.scatter(gmvp_b_lo['std']*100, gmvp_b_lo['return']*100, marker='*', s=200, c='red', zorder=5,
           label=f"GMVP beta ({gmvp_b_lo['std']:.2%})")

ax.axhline(y=rf*100, color='orange', linestyle='--', alpha=0.5, label=f'rf = {rf:.0%}')
ax.set_xlabel('Волатильность, %', fontsize=11)
ax.set_ylabel('Доходность, %', fontsize=11)
ax.set_title('Long-only: Historical vs Market Model', fontsize=13)
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# таблица сравнения ключевых портфелей
comparison = pd.read_parquet(PROCESSED / 'ef_beta_comparison_table.parquet')
comparison

In [ ]:
# динамика EF на beta
ef_dyn_beta_path = PROCESSED / 'ef_dynamics_beta_252d.pkl'
if ef_dyn_beta_path.exists():
    with open(ef_dyn_beta_path, 'rb') as f:
        ef_dyn_beta = pickle.load(f)
    plot_frontier_dynamics(ef_dyn_beta,
                          'Динамика EF (market model, rolling 252d)')
else:
    print('ef_dynamics_beta_252d.pkl не найден')

Market model фильтрует шумовые корреляции, оставляя только связь через рыночный фактор. Все off-diagonal корреляции в Sigma_beta положительные (все бета > 0), тогда как в исторической матрице есть отрицательные корреляции (PLZL -- контрциклический актив). Это может улучшить out-of-sample стабильность, но ценой потери реальных диверсификационных возможностей.

---
## 7. Задача 16 -- Ковариационная матрица на adjusted beta

Та же формула, что для задачи 13, но вместо $\beta_{hist}$ используем $\beta_{adj}$ (Blume). Идиосинкратические остатки ($\sigma^2_\varepsilon$) и рыночная дисперсия ($\sigma^2_m$) берутся те же, что и для исторической беты.

In [ ]:
# задача 16: Sigma на adjusted beta
# формула: Sigma_adj = beta_adj * beta_adj' * sigma2_m + diag(sigma2_eps)
# sigma2_eps и sigma2_m те же, что для historical beta

beta_adj = beta_adj_df.set_index('ticker').loc[tickers, 'beta_adjusted'].values

# sigma2_m_annual и sigma2_eps_annual уже рассчитаны выше (секция 6)

# ковариационная матрица на adjusted beta
Sigma_adj = np.outer(beta_adj, beta_adj) * sigma2_m_annual + np.diag(sigma2_eps_annual)

# проверки
print(f'Symmetric: {np.allclose(Sigma_adj, Sigma_adj.T)}')
print(f'PSD: {np.all(np.linalg.eigvalsh(Sigma_adj) > -1e-12)}')
print(f'Condition number: {np.linalg.cond(Sigma_adj):.1f}')

# сравнение с Sigma_beta (hist)
diff_frob = np.linalg.norm(Sigma_adj - Sigma_beta, 'fro')
rel_frob = diff_frob / np.linalg.norm(Sigma_beta, 'fro')
print(f'Frobenius distance (Sigma_adj vs Sigma_beta): {diff_frob:.6f} ({rel_frob:.1%})')

---
## 8. Задача 17 -- EF на Sigma_adj (adjusted beta)

Строим EF с ковариационной матрицей на скорректированных бетах. Ожидаемые доходности (mu) те же, что в задачах 4-8.

In [ ]:
# задача 17: EF на Sigma_adj

# unrestricted
gmvp_adj_unr = find_gmvp(mu, Sigma_adj, bounds=None)
mu_max_adj_unr = gmvp_adj_unr['return'] + 2 * (mu.max() - gmvp_adj_unr['return'])

ef_adj_unr, w_adj_unr, kp_adj_unr = build_efficient_frontier(
    mu, Sigma_adj, rf, n_points=200, bounds=None, mu_max=mu_max_adj_unr
)
print(f"Unrestricted GMVP: ret={kp_adj_unr['gmvp']['return']:.4f}, "
      f"std={kp_adj_unr['gmvp']['std']:.4f}")

# long-only
bounds_lo = [(0, 1)] * n
ef_adj_lo, w_adj_lo, kp_adj_lo = build_efficient_frontier(
    mu, Sigma_adj, rf, n_points=200, bounds=bounds_lo, mu_max=mu.max()
)
print(f"Long-only GMVP: ret={kp_adj_lo['gmvp']['return']:.4f}, "
      f"std={kp_adj_lo['gmvp']['std']:.4f}")

# график
fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(ef_adj_unr['portfolio_std']*100, ef_adj_unr['portfolio_return']*100,
        'b-', linewidth=2, label='Unrestricted (Sigma_adj)')
ax.plot(ef_adj_lo['portfolio_std']*100, ef_adj_lo['portfolio_return']*100,
        'g-', linewidth=2, label='Long-only (Sigma_adj)')
ax.scatter(kp_adj_unr['gmvp']['std']*100, kp_adj_unr['gmvp']['return']*100,
           marker='*', s=200, c='blue', zorder=5, label='GMVP unr')
ax.scatter(kp_adj_lo['gmvp']['std']*100, kp_adj_lo['gmvp']['return']*100,
           marker='*', s=200, c='green', zorder=5, label='GMVP lo')
ax.axhline(y=rf*100, color='orange', linestyle='--', alpha=0.5, label=f'rf = {rf:.0%}')
ax.set_xlabel('Волатильность, %', fontsize=12)
ax.set_ylabel('Доходность, %', fontsize=12)
ax.set_title('EF на ковариационной матрице adjusted beta', fontsize=14)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. Задача 18* -- Динамика EF на adjusted beta

Для каждой контрольной даты (year-end) прогоняем OLS, считаем adjusted beta, строим Sigma_adj и EF. Аналог задачи 15, но на скорректированных бетах.

In [ ]:
# задача 18: динамика EF на adjusted beta
# загружаем beta_dynamics для контрольных дат

beta_dyn_path = PROCESSED / 'beta_dynamics_rolling_252d.pkl'
if beta_dyn_path.exists():
    with open(beta_dyn_path, 'rb') as f:
        beta_dynamics = pickle.load(f)

    # загружаем rolling means для mu
    rolling_means = pd.read_parquet(PROCESSED / 'rolling_252d_means.parquet')

    # rf из risk_free_rate.parquet
    rf_raw = pd.read_parquet(RAW_DIR / 'risk_free_rate.parquet')
    rf_series = rf_raw.set_index('date')['rate_annual']
    rf_series.index = pd.to_datetime(rf_series.index)

    control_dates = beta_dynamics['dates']
    betas_dyn = beta_dynamics['betas']           # (n_dates, 30)
    sigma_eps_dyn = beta_dynamics['sigma_eps']    # (n_dates, 30)
    sigma2_m_dyn = beta_dynamics['sigma2_m']      # (n_dates,)

    adj_frontiers = []
    adj_kps = []
    adj_dates = []

    for di, d in enumerate(control_dates):
        date_ts = pd.Timestamp(d)
        # adjusted beta
        b_adj = (2/3) * betas_dyn[di] + (1/3)
        s2m_ann = sigma2_m_dyn[di] * 252
        s2e_ann = sigma_eps_dyn[di] ** 2 * 252

        Sigma_i = np.outer(b_adj, b_adj) * s2m_ann + np.diag(s2e_ann)

        if date_ts in rolling_means.index:
            mu_i = rolling_means.loc[date_ts].values
        else:
            # ближайшая дата
            idx = rolling_means.index.get_indexer([date_ts], method='nearest')[0]
            mu_i = rolling_means.iloc[idx].values

        # rf на эту дату
        mask = rf_series.index <= date_ts
        rf_i = rf_series[mask].iloc[-1] if mask.any() else rf_series.iloc[0]

        gmvp_i = find_gmvp(mu_i, Sigma_i, bounds=None)
        max_mu = np.max(mu_i)
        mu_max_i = gmvp_i['return'] + 2 * (max_mu - gmvp_i['return'])
        if mu_max_i <= gmvp_i['return']:
            mu_max_i = gmvp_i['return'] + 0.05

        try:
            ef_i, _, kp_i = build_efficient_frontier(
                mu_i, Sigma_i, rf_i, n_points=100, bounds=None, mu_max=mu_max_i
            )
            adj_frontiers.append(ef_i)
            adj_kps.append(kp_i)
            adj_dates.append(d)
        except Exception as e:
            print(f'Ошибка для {d}: {e}')

    # результат
    ef_dyn_adj = {
        'dates': adj_dates,
        'frontiers': adj_frontiers,
        'key_portfolios': adj_kps,
    }

    plot_frontier_dynamics(ef_dyn_adj,
                          'Динамика EF (adjusted beta, rolling 252d)')
else:
    print('beta_dynamics_rolling_252d.pkl не найден')

---
## 10. Задача 19 -- Сравнение трёх подходов к оценке Sigma

Наложение трёх границ:
1. **Historical Sigma** (rolling 252d) -- задачи 4-8
2. **Market Model Sigma_beta** (historical beta) -- задачи 13-14
3. **Market Model Sigma_adj** (adjusted beta) -- задачи 16-17

Сравнение по GMVP, tangency, Sharpe ratio.

In [ ]:
# задача 19: сравнение трёх подходов

fig, ax = plt.subplots(figsize=(14, 10))

# 1. Historical
ax.plot(ef_unrestricted['portfolio_std']*100, ef_unrestricted['portfolio_return']*100,
        'b-', linewidth=2, label='Historical Sigma', alpha=0.9)
# 2. Beta (historical)
ax.plot(ef_beta_unr['portfolio_std']*100, ef_beta_unr['portfolio_return']*100,
        'r-', linewidth=2, label='Sigma_beta (hist)', alpha=0.9)
# 3. Beta (adjusted)
ax.plot(ef_adj_unr['portfolio_std']*100, ef_adj_unr['portfolio_return']*100,
        'g--', linewidth=2, label='Sigma_adj (Blume)', alpha=0.9)

# GMVP точки
for label, gmvp_i, color, marker in [
    ('GMVP hist', w_hist_unr['gmvp'], 'blue', '*'),
    ('GMVP beta', w_beta['unrestricted']['gmvp'], 'red', '*'),
    ('GMVP adj', kp_adj_unr['gmvp'], 'green', '*'),
]:
    ax.scatter(gmvp_i['std']*100, gmvp_i['return']*100,
               marker=marker, s=200, c=color, zorder=5, label=label)

ax.axhline(y=rf*100, color='orange', linestyle='--', alpha=0.5, label=f'rf = {rf:.0%}')
ax.scatter(vol_stocks*100, mu*100, c='gray', s=20, alpha=0.5, zorder=3)

ax.set_xlabel('Годовая волатильность, %', fontsize=13)
ax.set_ylabel('Годовая доходность, %', fontsize=13)
ax.set_title('Сравнение трёх подходов к оценке Sigma (unrestricted)', fontsize=15)
ax.legend(fontsize=10, loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# таблица сравнения GMVP
rows_cmp = []
for name, gmvp_i in [
    ('Historical', w_hist_unr['gmvp']),
    ('Beta (hist)', w_beta['unrestricted']['gmvp']),
    ('Beta (adj)', kp_adj_unr['gmvp']),
]:
    rows_cmp.append({
        'Метод': name,
        'Return, %': f"{gmvp_i['return']*100:.2f}",
        'Std, %': f"{gmvp_i['std']*100:.2f}",
        'Sharpe': f"{(gmvp_i['return']-rf)/gmvp_i['std']:.4f}",
    })
pd.DataFrame(rows_cmp)

---
## 11. Задача 20* -- Сравнение для разных окон оценки

Сравнение траекторий GMVP для трёх подходов (historical, beta_hist, beta_adj) на разных контрольных датах.

In [ ]:
# задача 20: три траектории GMVP
# нужны ef_dynamics для каждого подхода

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# historical (rolling 252d)
dates_h = ef_dyn_rolling['dates']
gmvp_ret_h = [kp['gmvp']['return']*100 for kp in ef_dyn_rolling['key_portfolios']]
gmvp_std_h = [kp['gmvp']['std']*100 for kp in ef_dyn_rolling['key_portfolios']]
ax1.plot(dates_h, gmvp_ret_h, 'bo-', markersize=5, label='Historical')
ax2.plot(dates_h, gmvp_std_h, 'bo-', markersize=5, label='Historical')

# beta (historical) -- из ef_dynamics_beta_252d
if ef_dyn_beta_path.exists():
    dates_b = ef_dyn_beta['dates']
    gmvp_ret_b = [kp['gmvp']['return']*100 for kp in ef_dyn_beta['key_portfolios']]
    gmvp_std_b = [kp['gmvp']['std']*100 for kp in ef_dyn_beta['key_portfolios']]
    ax1.plot(dates_b, gmvp_ret_b, 'ro-', markersize=5, label='Beta (hist)')
    ax2.plot(dates_b, gmvp_std_b, 'ro-', markersize=5, label='Beta (hist)')

# beta (adjusted) -- из задачи 18
if 'ef_dyn_adj' in dir() and len(adj_dates) > 0:
    gmvp_ret_a = [kp['gmvp']['return']*100 for kp in adj_kps]
    gmvp_std_a = [kp['gmvp']['std']*100 for kp in adj_kps]
    ax1.plot(adj_dates, gmvp_ret_a, 'gs-', markersize=5, label='Beta (adj)')
    ax2.plot(adj_dates, gmvp_std_a, 'gs-', markersize=5, label='Beta (adj)')

ax1.set_ylabel('Доходность GMVP, %')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

ax2.set_ylabel('Волатильность GMVP, %')
ax2.set_xlabel('Дата')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

fig.suptitle('Траектория GMVP: сравнение трёх подходов', fontsize=14)
plt.tight_layout()
plt.show()

---
## 12. Задачи 21-25 -- Бонусные задачи

Заготовки для дополнительных задач.

### Задача 22 -- проверка Black's two-fund theorem

Проверяем теорему Блэка (1972): любой портфель на unrestricted MVF есть линейная комбинация любых двух других портфелей с этой же границы. Базисная пара -- GMVP и tangency. Дополнительно проводим аналитическую проверку по формуле Мертона (1972) и показываем нарушение теоремы для constrained случаев.

In [ ]:
# Задача 22: проверка two-fund theorem Блэка (1972)
# Базисная пара: GMVP и tangency (unrestricted)

w_gmvp = w_hist_unr['gmvp']['weights']
w_tang = w_hist_unr['tangency']['weights']
frontier_weights = w_hist_unr['frontier_weights']

# для каждого портфеля на EF ищем alpha: w_i = alpha * w_gmvp + (1 - alpha) * w_tang
delta_w = w_gmvp - w_tang
delta_dot = delta_w @ delta_w

results = []
for i, w_i in enumerate(frontier_weights):
    alpha_i = (w_i - w_tang) @ delta_w / delta_dot
    w_recon = alpha_i * w_gmvp + (1 - alpha_i) * w_tang

    err_l2 = np.linalg.norm(w_i - w_recon)
    err_ret = abs(w_i @ mu - w_recon @ mu)
    err_std = abs(np.sqrt(w_i @ Sigma @ w_i) - np.sqrt(w_recon @ Sigma @ w_recon))

    results.append({
        'target_return': w_i @ mu,
        'alpha': alpha_i,
        'err_l2': err_l2,
        'err_return': err_ret,
        'err_std': err_std,
    })

twofund_df = pd.DataFrame(results)
print(f"Max ошибка реконструкции (L2): {twofund_df['err_l2'].max():.2e}")
print(f"Mean ошибка: {twofund_df['err_l2'].mean():.2e}")
print(f"Max ошибка доходности: {twofund_df['err_return'].max():.2e}")
print(f"Max ошибка std: {twofund_df['err_std'].max():.2e}")

threshold = 1e-4
all_below = (twofund_df['err_l2'] < threshold).all()
print(f"\nВсе ошибки < {threshold}: {'Да' if all_below else 'Нет'}")
print("=> Two-fund theorem ПОДТВЕРЖДЕНА для unrestricted MVF")

# аналитическая проверка (Merton 1972): w(mu_p) = Sigma_inv @ (lambda1 * mu + lambda2 * 1)
Sigma_inv = np.linalg.inv(Sigma)
ones = np.ones(len(mu))
A_m = ones @ Sigma_inv @ mu
B_m = mu @ Sigma_inv @ mu
C_m = ones @ Sigma_inv @ ones
D_m = B_m * C_m - A_m**2

# линейность: w = a + b * mu_p
a_vec = Sigma_inv @ ((B_m / D_m) * ones - (A_m / D_m) * mu)
b_vec = Sigma_inv @ ((C_m / D_m) * mu - (A_m / D_m) * ones)

target_rets = twofund_df['target_return'].values
w_analytical = np.array([a_vec + b_vec * r for r in target_rets])
max_linearity = max(np.linalg.norm(w_analytical[i] - (a_vec + b_vec * target_rets[i]))
                     for i in range(len(target_rets)))
print(f"\nАналитика (Merton): D = {D_m:.2f} > 0, линейность error = {max_linearity:.2e}")

# two-fund на аналитических весах (machine precision)
w1_a, w2_a = w_analytical[0], w_analytical[-1]
dw_a = w1_a - w2_a
dd_a = dw_a @ dw_a
max_err_a = max(np.linalg.norm(w_analytical[i] -
    ((w_analytical[i] - w2_a) @ dw_a / dd_a) * w1_a -
    (1 - (w_analytical[i] - w2_a) @ dw_a / dd_a) * w2_a)
    for i in range(len(w_analytical)))
print(f"Two-fund на аналит. весах: max L2 = {max_err_a:.2e} (machine epsilon)")

# сравнение analytical vs numerical
max_diff = np.max(np.abs(w_analytical - frontier_weights))
print(f"Max |w_analyt - w_numer|: {max_diff:.2e} (точность SLSQP)")

# --- визуализация ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) alpha vs return
ax = axes[0]
ax.plot(twofund_df['target_return'] * 100, twofund_df['alpha'], 'b-', linewidth=1.5)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
ax.axvspan(ax.get_xlim()[0], ax.get_xlim()[1], alpha=0.0)
r_gmvp = w_gmvp @ mu * 100
r_tang = w_tang @ mu * 100
ax.axvline(x=r_gmvp, color='red', linestyle=':', alpha=0.6, label='GMVP')
ax.axvline(x=r_tang, color='green', linestyle=':', alpha=0.6, label='Tangency')
ax.set_xlabel('Целевая доходность, %')
ax.set_ylabel('alpha')
ax.set_title('Two-fund: alpha(mu_p)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# 2) ошибка реконструкции
ax = axes[1]
ax.semilogy(range(len(twofund_df)), twofund_df['err_l2'], 'b-', alpha=0.7, label='L2 error')
ax.axhline(y=1e-4, color='green', linestyle='--', alpha=0.6, label='SLSQP precision')
ax.set_xlabel('Индекс портфеля')
ax.set_ylabel('Ошибка (L2)')
ax.set_title('Ошибка реконструкции весов')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# 3) EF: аналитическая vs численная + two-fund line
ax = axes[2]
ax.plot(ef_unrestricted['portfolio_std'] * 100, ef_unrestricted['portfolio_return'] * 100,
        'b-', linewidth=2, label='EF (SLSQP)', alpha=0.8)

# two-fund комбинации по всему диапазону alpha
alphas_line = np.linspace(-0.5, 1.5, 300)
tf_stds = [np.sqrt((a * w_gmvp + (1 - a) * w_tang) @ Sigma @ (a * w_gmvp + (1 - a) * w_tang)) * 100
           for a in alphas_line]
tf_rets = [(a * w_gmvp + (1 - a) * w_tang) @ mu * 100 for a in alphas_line]
ax.plot(tf_stds, tf_rets, 'r--', linewidth=1.5, label='Two-fund line', alpha=0.7)

ax.scatter([w_hist_unr['gmvp']['std'] * 100], [w_hist_unr['gmvp']['return'] * 100],
           marker='*', s=200, c='red', zorder=5, label='GMVP')
ax.scatter([w_hist_unr['tangency']['std'] * 100], [w_hist_unr['tangency']['return'] * 100],
           marker='D', s=100, c='green', zorder=5, label='Tangency')
ax.set_xlabel('Стандартное отклонение, %')
ax.set_ylabel('Доходность, %')
ax.set_title('Аналитическая vs численная EF')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# проверка нарушения для constrained
print("\n--- Constrained: нарушение two-fund theorem ---")
for regime_name, regime_key in [('long_only', 'long_only'), ('short_25', 'short_25')]:
    with open(PROCESSED / 'ef_portfolios.pkl', 'rb') as f:
        all_regimes = pickle.load(f)
    rd = all_regimes[regime_key]
    w1_c = rd['gmvp']['weights']
    w2_c = rd['tangency']['weights']
    fw_c = rd['frontier_weights']
    dw_c = w1_c - w2_c
    dd_c = dw_c @ dw_c
    errs = [np.linalg.norm(fw_c[i] - ((fw_c[i] - w2_c) @ dw_c / dd_c) * w1_c -
            (1 - (fw_c[i] - w2_c) @ dw_c / dd_c) * w2_c) for i in range(len(fw_c))]
    n_above = sum(1 for e in errs if e > 1e-6)
    print(f"  {regime_name}: max_err = {max(errs):.2e}, "
          f"портфелей с err > 1e-6: {n_above}/{len(fw_c)} => теорема НЕ выполняется")

### Задача 23 -- Monte Carlo граница портфелей

Генерация 500 000 случайных портфелей для трёх режимов ограничений (long-only, unrestricted, short ≤ 25%) методом Монте-Карло. Веса генерируются из распределения Дирихле (long-only), нормального с нормализацией (unrestricted) и rejection sampling (short ≤ 25%). MC-граница извлекается как верхняя огибающая облака (200 бинов по σ, max return в каждом). Аналитическая EF из задач 5–7 — верхняя граница облака: случайная диверсификация не может достичь оптимума.

In [ ]:
# Задача 23: загрузка предрассчитанных MC-результатов (500k портфелей, ~3 мин расчёта)

# MC-облака и границы
mc_clouds = {}
mc_frontiers = {}
regimes = ['long_only', 'unrestricted', 'short_25']
for regime in regimes:
    mc_clouds[regime] = pd.read_parquet(PROCESSED / f'step11_mc_portfolios_{regime}.parquet')
    mc_frontiers[regime] = pd.read_parquet(PROCESSED / f'step11_mc_frontier_{regime}.parquet')

with open(PROCESSED / 'step11_mc_key_portfolios.pkl', 'rb') as f:
    mc_key = pickle.load(f)

# Аналитические EF уже загружены (ef_long_only, ef_unrestricted, ef_short_25)
ef_data = {
    'long_only': ef_long_only,
    'unrestricted': ef_unrestricted,
    'short_25': ef_short_25,
}

# Аналитические ключевые портфели
ef_keys = {}
for regime in regimes:
    with open(PROCESSED / f'ef_{regime}_weights.pkl', 'rb') as f:
        ef_keys[regime] = pickle.load(f)

# -- Визуализация: 3 режима рядом --
titles = {'long_only': 'Long-only (N=500 000)',
          'unrestricted': 'Unrestricted (N=500 000)',
          'short_25': 'Short ≤ 25% (N=500 000)'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, regime in zip(axes, regimes):
    cloud = mc_clouds[regime]
    frontier = mc_frontiers[regime]
    ef = ef_data[regime]

    # облако MC
    sc = ax.scatter(cloud['portfolio_std'] * 100, cloud['portfolio_return'] * 100,
                    c=cloud['sharpe'], cmap='viridis', s=1, alpha=0.15, rasterized=True)

    # аналитическая EF
    ax.plot(ef['portfolio_std'] * 100, ef['portfolio_return'] * 100,
            'r-', lw=2, label='Аналитическая EF', zorder=5)

    # MC-граница (верхняя огибающая)
    ax.plot(frontier['portfolio_std'] * 100, frontier['portfolio_return'] * 100,
            '--', color='orange', lw=1.5, label='MC-граница', zorder=4)

    # ключевые точки -- аналитические
    g_an = ef_keys[regime]['gmvp']
    t_an = ef_keys[regime]['tangency']
    ax.scatter(g_an['std'] * 100, g_an['return'] * 100,
               marker='*', s=200, c='red', edgecolors='k', zorder=6, label='GMVP (аналит.)')
    ax.scatter(t_an['std'] * 100, t_an['return'] * 100,
               marker='*', s=200, c='green', edgecolors='k', zorder=6, label='Tangency (аналит.)')

    # ключевые точки -- MC
    g_mc = mc_key[regime]['gmvp']
    t_mc = mc_key[regime]['tangency']
    ax.scatter(g_mc['std'] * 100, g_mc['return'] * 100,
               marker='*', s=150, c='orange', edgecolors='k', zorder=6, label='GMVP (MC)')
    ax.scatter(t_mc['std'] * 100, t_mc['return'] * 100,
               marker='*', s=150, c='lime', edgecolors='k', zorder=6, label='Tangency (MC)')

    ax.set_xlabel('Стандартное отклонение, %')
    ax.set_ylabel('Ожидаемая доходность, %')
    ax.set_title(titles[regime])
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.3)

    # colorbar
    plt.colorbar(sc, ax=ax, label='Sharpe ratio', shrink=0.8)

plt.tight_layout()
plt.show()

# -- Сравнительная таблица MC vs аналитика --
rows = []
for regime in regimes:
    g_mc = mc_key[regime]['gmvp']
    g_an = ef_keys[regime]['gmvp']
    t_mc = mc_key[regime]['tangency']
    t_an = ef_keys[regime]['tangency']

    # для GMVP аналитического Sharpe считаем вручную (его нет в pkl)
    g_an_sharpe = (g_an['return'] - rf) / g_an['std']

    rows.append({
        'Режим': regime,
        'GMVP std MC, %': f"{g_mc['std']*100:.2f}",
        'GMVP std аналит., %': f"{g_an['std']*100:.2f}",
        'Gap GMVP std, %': f"{(g_mc['std']/g_an['std'] - 1)*100:+.1f}",
        'Tangency Sharpe MC': f"{t_mc['sharpe']:.3f}",
        'Tangency Sharpe аналит.': f"{t_an['sharpe']:.3f}",
        'Gap Sharpe, %': f"{(t_mc['sharpe']/t_an['sharpe'] - 1)*100:+.1f}",
    })

comparison_df = pd.DataFrame(rows)
print('Сравнение MC (500k) vs аналитическая EF:')
display(comparison_df)

# -- Сходимость MC --
conv = pd.read_parquet(PROCESSED / 'step11_mc_convergence.parquet')
print(f'\nСходимость MC (long-only): при N={int(conv["n_simulations"].iloc[-1]):,} '
      f'gap GMVP std = {conv["gmvp_std_gap"].iloc[-1]:.1%}, '
      f'gap max Sharpe = {conv["max_sharpe_gap"].iloc[-1]:.1%}, '
      f'mean frontier distance = {conv["mean_frontier_distance"].iloc[-1]:.4f}')

### Задача 24 -- Портфель максимального риска

Портфель с максимальной волатильностью при заданных ограничениях. Это нижняя часть "пули" на графике (ниже GMVP).

In [ ]:
# задача 24: портфель максимального риска (long-only)

def find_max_risk(mu, cov, bounds=None):
    """Портфель с максимальной дисперсией."""
    n_a = len(mu)
    x0 = np.ones(n_a) / n_a

    def neg_variance(w):
        return -(w @ cov @ w)

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    result = minimize(neg_variance, x0, method='SLSQP',
                      bounds=bounds, constraints=constraints,
                      options={'ftol': 1e-12, 'maxiter': 1000})
    w = result.x
    return {
        'weights': w,
        'return': w @ mu,
        'std': np.sqrt(w @ cov @ w),
        'success': result.success,
    }

max_risk_lo = find_max_risk(mu, Sigma, bounds=[(0,1)]*n)
print(f"Max risk (long-only): ret={max_risk_lo['return']*100:.2f}%, "
      f"std={max_risk_lo['std']*100:.2f}%")

# топ-5 по весу
idx_sorted = np.argsort(max_risk_lo['weights'])[::-1]
print('\nТоп-5 акций по весу:')
for rank, idx in enumerate(idx_sorted[:5]):
    print(f"  {rank+1}. {tickers[idx]:>8s}: {max_risk_lo['weights'][idx]*100:.1f}%")

### Задача 25 -- Implementation shortfall

Оценка разрыва между теоретической доходностью оптимального портфеля и реальной, с учётом комиссий, проскальзывания и задержки ребалансировки. Требует дополнительных данных по торговым объёмам и спредам -- выходит за рамки текущего датасета.